# Sentinel AI — GLOVE model retrain (kill the wall false positives)

The glove model scores a **wall (~0.63) higher than a real glove (~0.57)**, so no
threshold can separate them. This retrain fixes it at the source by feeding the model
**hard negatives from your actual room** (the wall it hallucinates on), labelled as
*'no glove here'*, plus CLAHE-baked positives to match your live preprocessing.

**Set `Runtime ▸ Change runtime type ▸ GPU (T4)` first.**

### One-time upload to Google Drive (My Drive, top level)
1. `gloves_retrain_v3.zip`  — zip your `Training Work/gloves_retrain_v3` folder.
2. `glove_hard_negatives.zip` — already prepared for you in `retraining/`.

## 0. GPU check + install

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics albumentations

## 1. Mount Drive & unzip both datasets

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, zipfile, glob, shutil, cv2

def unzip(z, dst):
    assert os.path.exists(z), f'Missing: {z}'
    with zipfile.ZipFile(z) as f: f.extractall(dst)

unzip('/content/drive/MyDrive/gloves_retrain_v3.zip', '/content/glove_base')
unzip('/content/drive/MyDrive/glove_hard_negatives.zip', '/content/hardneg')
# handle a possible nested top folder in the base zip
BASE='/content/glove_base'
if not os.path.isdir(f'{BASE}/train'):
    sub=[d for d in glob.glob(f'{BASE}/*') if os.path.isdir(d)]
    if len(sub)==1: BASE=sub[0]
print('base:', BASE, '| train imgs:', len(os.listdir(f'{BASE}/train/images')))
print('hard negatives:', len(os.listdir('/content/hardneg/images')))

## 2. CLAHE-bake the base positives
Applies the SAME CLAHE (clipLimit=2.0, 8x8) your live engine uses so training matches
inference. Writes to `/content/glove_clahe`. (Hard negatives are already CLAHE — added
in the next cell as-is, not re-baked.)

In [ ]:
DST='/content/glove_clahe'
clahe=cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
EXT=('.jpg','.jpeg','.png','.bmp','.webp')
def bake(p,out):
    im=cv2.imread(p)
    if im is None: return False
    lab=cv2.cvtColor(im,cv2.COLOR_BGR2LAB); l,a,b=cv2.split(lab); l=clahe.apply(l)
    cv2.imwrite(out, cv2.cvtColor(cv2.merge([l,a,b]),cv2.COLOR_LAB2BGR)); return True
for split in ['train','valid','test']:
    si=f'{BASE}/{split}/images'
    if not os.path.isdir(si): continue
    di=f'{DST}/{split}/images'; dl=f'{DST}/{split}/labels'
    os.makedirs(di,exist_ok=True); os.makedirs(dl,exist_ok=True)
    for lf in glob.glob(f'{BASE}/{split}/labels/*.txt'):
        shutil.copy2(lf, f'{dl}/{os.path.basename(lf)}')
    imgs=[p for p in glob.glob(f'{si}/*') if p.lower().endswith(EXT)]
    for p in imgs: bake(p, f'{di}/{os.path.basename(p)}')
    print(f'{split}: baked {len(imgs)}')

## 3. Merge the room hard-negatives into the TRAIN split
These teach the model *'this wall / this room is NOT a glove.'*

In [ ]:
di=f'{DST}/train/images'; dl=f'{DST}/train/labels'
added=0
for p in glob.glob('/content/hardneg/images/*'):
    shutil.copy2(p, f'{di}/{os.path.basename(p)}')
    stem=os.path.splitext(os.path.basename(p))[0]
    open(f'{dl}/{stem}.txt','w').close()   # empty label = background
    added+=1
print('hard negatives merged into train:', added)

with open(f'{DST}/data.yaml','w') as f:
    f.write('path: /content/glove_clahe\ntrain: train/images\nval: valid/images\ntest: test/images\n'
            'nc: 1\nnames:\n  0: Glove\n')
print('data.yaml written')

## 4. Train
`hsv_v`/`hsv_h` handle your dim/green room. ~30–60 min on a T4 for this size.

In [ ]:
from ultralytics import YOLO
model=YOLO('yolov8s.pt')
model.train(
    data=f'{DST}/data.yaml', epochs=120, imgsz=640, batch=16,
    hsv_h=0.03, hsv_s=0.7, hsv_v=0.5,
    mosaic=1.0, mixup=0.1, degrees=5, translate=0.1, scale=0.5, erasing=0.4,
    name='glove_v4_hardneg')

## 5. Save the new weights to Drive

In [ ]:
import glob, os, shutil
# find the newest trained model regardless of folder suffix (glove_v4_hardneg, -2, ...)
best=max(glob.glob('/content/runs/detect/glove_v4_hardneg*/weights/best.pt'), key=os.path.getmtime)
print('found:', best)
shutil.copy(best,'/content/drive/MyDrive/glove_v4_hardneg_best.pt')
print('Saved to My Drive as glove_v4_hardneg_best.pt')

## 6. Deploy on the server
1. Download `glove_v4_hardneg_best.pt`.
2. Put it at `Training Work/glove_model_v4/weights/best.pt` and point the V2 path in
   `ai-engine/main.py` (line ~79, `GLOVE_V2_PATH`) at it — **or** just overwrite
   `Training Work/glove_model_v2/weights/best.pt` with it.
3. Restart the AI engine.
4. Point the camera at the wall — it should now stay **0 gloves** (wall ≈ 0.05),
   while a worn glove still registers.